In [1]:
import json
import sys
from datetime import datetime, timezone
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction import DictVectorizer
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

In [2]:
start_folder = Path.cwd()

project_folder = None

for folder in (start_folder, *start_folder.parents):
    for path in (folder / "app" / "flatfairproj", folder / "flatfairproj", folder):
        if (path / "manage.py").is_file():
            project_folder = path

            break

    if project_folder is not None:
        break

if project_folder is None:
    raise Exception("Run this notebook from inside the FlatFair project folder.")

root_folder = project_folder.parents[1]

training_folder = project_folder / "assets" / "training"

for folder in (training_folder, project_folder):
    if str(folder) not in sys.path:
        sys.path.insert(0, str(folder))

from pretrained import FEATURE_NAMES, MODEL_VERSION, PACKAGE_TIERS, packTraits
from write_manifest import file_hash

In [3]:
data_file = training_folder / "ResaleflatpricesbasedonregistrationdatefromJan2017onwards.csv"

short_run = True

run_folder = training_folder / "check_runs" / "package"

if short_run:
    model_file = run_folder / "package_model.joblib"

    report_folder = run_folder

    row_count = 4000
else:
    model_file = project_folder / "assets" / "ml" / "package_model.joblib"

    report_folder = training_folder

    row_count = 18000

metrics_file = report_folder / "package_model_metrics.json"

check_file = report_folder / "package_model_eval.json"

train_options = {
    "data_file": data_file,
    "rows": row_count,
    "seed": 2108,
    "model_file": model_file,
    "metrics_file": metrics_file,
}

check_options = {
    "data_file": data_file,
    "rows": row_count,
    "seed": 2109,
    "model_file": model_file,
    "metrics_file": metrics_file,
    "check_file": check_file,
}

print("Short training check" if short_run else "Full training")
print({"rows": train_options["rows"], "seed": train_options["seed"], "output": str(model_file)})

Short training check
{'rows': 4000, 'seed': 2108, 'output': '/home/coco/Desktop/c_windows/Users/Koko/Desktop/HELP/25_10_2026/IS2108/flatfair/.flatfair-local/short-names-20261114T144349Z/notebook-after/app/training/check_runs/package/package_model.joblib'}


In [4]:
print("Saved results from an earlier run")

saved_results = json.loads((training_folder / "package_model_metrics.json").read_text())

print({name: saved_results[name] for name in ["rows", "accuracy", "macroF1", "syntheticLabels"]})

Saved results from an earlier run
{'rows': 18000, 'accuracy': 0.8515555555555555, 'macroF1': 0.850890164302363, 'syntheticLabels': True}


In [5]:
def rule_score(values, days, threshold):
    score = 0.0

    if values["bandPosition"] > 1:
        score += 1.3

    if values["askingRatio"] > 1.18:
        score += 0.7

    if values["townTransactions"] >= threshold:
        score += 1.0

    if days >= 30:
        score += 0.7

    if days >= 75:
        score += 0.7

    if values["leaseRemaining"] < 50:
        score += 0.4

    if values["askingRatio"] < 0.98:
        score -= 0.5

    return score


def rule_label(score):
    if score < 0.6:
        return "none"

    if score < 1.6:
        return "boost30"

    if score < 2.8:
        return "boost90"

    return "topSpot"

In [6]:
def rule_data(data, seed, rows):
    rng = np.random.default_rng(seed)

    data = data.drop_duplicates().copy()

    months = pd.PeriodIndex(data.month, freq="M")

    last_month = months.max()

    recent = data[months >= last_month - 11]

    town_nums = {}

    for town, count in recent.town.value_counts().items():
        town_nums[town] = int(count)

    cutoff = float(np.quantile(list(town_nums.values()), 0.65))

    source = data[months >= last_month - 35]

    source = source[source.flat_type.isin(["2 ROOM", "3 ROOM", "4 ROOM", "5 ROOM", "EXECUTIVE"])]

    source = source.sample(n=min(rows, len(source)), random_state=seed).reset_index(drop=True)

    year_now = last_month.year + (last_month.month - 1) / 12

    trait_rows = []

    rule_tags = []

    for row in source.itertuples():
        worth = float(row.resale_price) * rng.lognormal(0, 0.03)

        usual = rng.choice([0.96, 1.03, 1.13, 1.24], p=[0.35, 0.35, 0.22, 0.08])

        ratio = rng.normal(usual, 0.04)

        low = worth * rng.uniform(0.88, 0.95)

        high = worth * rng.uniform(1.05, 1.14)

        days = int(min(180, rng.exponential(30)))

        storeys = [int(part) for part in row.storey_range.split(" TO ")]

        mid_floor = sum(storeys) / 2

        ask = worth * ratio

        town_sales = town_nums.get(row.town, 0)

        values = packTraits(
            row.town, row.flat_type, row.floor_area_sqm, mid_floor,
            row.lease_commence_date, ask, worth, low, high, days,
            year_now, town_sales,
        )

        score = rule_score(values, days, cutoff)

        score += rng.normal(0, 0.22)

        label = rule_label(score)

        if rng.random() < 0.04:
            label = rng.choice(list(PACKAGE_TIERS))

        trait_rows.append(values)

        rule_tags.append(label)

    return trait_rows, rule_tags, town_nums, cutoff, str(last_month)

In [7]:
def count_each(values, class_names):
    values = np.asarray(values)

    counts = {}

    for label in class_names:
        counts[label] = int(np.sum(values == label))

    return counts


def fit_model(options):
    data = pd.read_csv(options["data_file"])

    trait_rows, rule_tags, town_nums, cutoff, last_month = rule_data(
        data, options["seed"], options["rows"],
    )

    train_rows, test_rows, train_tags, test_tags = train_test_split(
        trait_rows, rule_tags, test_size=0.25, random_state=options["seed"], stratify=rule_tags,
    )

    forest = RandomForestClassifier(
        n_estimators=64, max_depth=12, min_samples_leaf=10,
        max_features=0.8, n_jobs=2, random_state=options["seed"],
    )

    model = Pipeline([
        ("vectorizer", DictVectorizer(sparse=False)),
        ("classifier", forest),
    ])

    model.fit(train_rows, train_tags)

    guesses = model.predict(test_rows)

    kind_names = list(PACKAGE_TIERS)

    test_nums = count_each(test_tags, kind_names)

    metrics = {
        "version": MODEL_VERSION,
        "syntheticLabels": True,
        "modelType": "RandomForestClassifier",
        "seed": options["seed"],
        "source": Path(options["data_file"]).name,
        "sourceMaxMonth": last_month,
        "rows": len(trait_rows),
        "trainRows": len(train_rows),
        "testRows": len(test_rows),
        "split": "Stratified 75/25 holdout; no tuning or refitting on test rows",
        "accuracy": float(accuracy_score(test_tags, guesses)),
        "macroF1": float(f1_score(test_tags, guesses, average="macro")),
        "majorityBaselineAccuracy": max(test_nums.values()) / len(test_tags),
        "classNames": kind_names,
        "confusionMatrix": confusion_matrix(test_tags, guesses, labels=kind_names).tolist(),
        "classBalance": {
            "all": count_each(rule_tags, kind_names),
            "train": count_each(train_tags, kind_names),
            "test": test_nums,
        },
        "competitionThreshold": cutoff,
        "features": FEATURE_NAMES,
    }

    ai_data = {
        "version": MODEL_VERSION,
        "features": FEATURE_NAMES,
        "encodedFeatures": model.named_steps["vectorizer"].get_feature_names_out().tolist(),
        "model": model,
        "townTransactions": town_nums,
        "competitionThreshold": cutoff,
        "sourceMaxMonth": last_month,
        "syntheticLabels": True,
    }

    return {"model_data": ai_data, "metrics": metrics}

In [8]:
def grade_tags(truth, predictions, class_names):
    return {
        "accuracy": float(accuracy_score(truth, predictions)),
        "perClass": classification_report(truth, predictions, labels=class_names, output_dict=True, zero_division=0),
        "confusionMatrix": confusion_matrix(truth, predictions, labels=class_names).tolist(),
    }


def try_model(options):
    train_set = json.loads(Path(options["metrics_file"]).read_text())

    if options["seed"] == train_set["seed"]:
        raise Exception("Use a fresh seed, distinct from the recorded training seed.")

    ai_path = Path(options["model_file"])

    ai_data = joblib.load(ai_path)

    model = ai_data["model"]

    model.named_steps["classifier"].n_jobs = 1

    data = pd.read_csv(options["data_file"])

    trait_rows, noisy_tags, town_nums, cutoff, last_month = rule_data(
        data, options["seed"], options["rows"],
    )

    if town_nums != ai_data["townTransactions"] or cutoff != ai_data["competitionThreshold"]:
        raise Exception("Evaluation source activity must match the trained policy context.")

    clean_tags = []

    for values in trait_rows:
        clean_mark = rule_score(values, values["daysListed"], ai_data["competitionThreshold"])

        clean_tags.append(rule_label(clean_mark))

    guesses = model.predict(trait_rows)

    kind_names = list(PACKAGE_TIERS)

    code_names = model.named_steps["vectorizer"].get_feature_names_out()

    powers = model.named_steps["classifier"].feature_importances_

    grouped = {name: 0.0 for name in ai_data["features"]}

    for code_name, power in zip(code_names, powers):
        trait = code_name.split("=", 1)[0]

        grouped[trait] += float(power)

    trait_powers = dict(sorted(grouped.items(), key=lambda item: item[1], reverse=True))

    code_powers = dict(
        sorted(zip(code_names.tolist(), powers.tolist()), key=lambda item: item[1], reverse=True)
    )

    return {
        "evaluatedAt": datetime.now(timezone.utc).isoformat(),
        "version": ai_data["version"],
        "artifactSha256": file_hash(ai_path),
        "seed": options["seed"],
        "trainingSeed": train_set["seed"],
        "rows": len(trait_rows),
        "sourceMaxMonth": last_month,
        "classNames": kind_names,
        "competitionThreshold": cutoff,
        "sampling": "Fresh synthetic valuation, asking, age and label noise draws from unchanged training generator with a different seed; underlying HDB property rows may overlap training.",
        "noiseFreePolicy": "Exact generating score and class cutoffs, with Gaussian score noise and random label replacement removed.",
        "againstPolicy": grade_tags(clean_tags, guesses, kind_names),
        "againstNoisyLabels": grade_tags(noisy_tags, guesses, kind_names),
        "policyAgainstNoisyLabelsAccuracy": float(accuracy_score(noisy_tags, clean_tags)),
        "featureImportanceMethod": "Random forest mean decrease in impurity; categorical one-hot importances summed to original feature. Descriptive, not causal, and potentially biased toward high-cardinality inputs.",
        "featureImportances": trait_powers,
        "encodedFeatureImportances": code_powers,
        "limitations": "Checks learning of the assumed synthetic policy, not real-world package benefit, sales lift or independent future market performance. No model fitting or artifact writes occur.",
    }

In [9]:
def save_model(result, options):
    ai_path = Path(options["model_file"])

    ai_path.parent.mkdir(parents=True, exist_ok=True)

    joblib.dump(result["model_data"], ai_path, compress=3)

    Path(options["metrics_file"]).write_text(json.dumps(result["metrics"], indent=2) + "\n")

    return {**result["metrics"], "artifactBytes": ai_path.stat().st_size}


def save_check(report, options):
    Path(options["check_file"]).write_text(json.dumps(report, indent=2) + "\n")

    return {
        "rows": report["rows"],
        "policyAgreement": report["againstPolicy"]["accuracy"],
        "noisyLabelAccuracy": report["againstNoisyLabels"]["accuracy"],
        "featureImportances": report["featureImportances"],
    }

In [10]:
train_result = fit_model(train_options)

In [11]:
train_summary = save_model(train_result, train_options)

print(json.dumps(train_summary, indent=2))

{
  "version": "package-v1",
  "syntheticLabels": true,
  "modelType": "RandomForestClassifier",
  "seed": 2108,
  "source": "ResaleflatpricesbasedonregistrationdatefromJan2017onwards.csv",
  "sourceMaxMonth": "2026-10",
  "rows": 4000,
  "trainRows": 3000,
  "testRows": 1000,
  "split": "Stratified 75/25 holdout; no tuning or refitting on test rows",
  "accuracy": 0.853,
  "macroF1": 0.8515843434166772,
  "majorityBaselineAccuracy": 0.359,
  "classNames": [
    "none",
    "boost30",
    "boost90",
    "topSpot"
  ],
  "confusionMatrix": [
    [
      279,
      27,
      1,
      1
    ],
    [
      34,
      292,
      31,
      2
    ],
    [
      4,
      18,
      193,
      13
    ],
    [
      4,
      4,
      8,
      89
    ]
  ],
  "classBalance": {
    "all": {
      "none": 1232,
      "boost30": 1434,
      "boost90": 913,
      "topSpot": 421
    },
    "train": {
      "none": 924,
      "boost30": 1075,
      "boost90": 685,
      "topSpot": 316
    },
    "test": 

In [12]:
check_report = try_model(check_options)

In [13]:
check_summary = save_check(check_report, check_options)

print(json.dumps(check_summary, indent=2))

{
  "rows": 4000,
  "policyAgreement": 0.95675,
  "noisyLabelAccuracy": 0.84925,
  "featureImportances": {
    "daysListed": 0.26613593649740985,
    "townTransactions": 0.2472705154446071,
    "bandPosition": 0.22959003005337422,
    "askingRatio": 0.18262130171407562,
    "leaseRemaining": 0.03004496929679476,
    "estimatedPrice": 0.017944714563142835,
    "floorAreaSqm": 0.012606483609362314,
    "storey": 0.005748464503777604,
    "town": 0.0050124855750975205,
    "flatType": 0.0030250987423582685
  }
}


In [14]:
saved_model = joblib.load(model_file)

fresh_results = json.loads(metrics_file.read_text())

assert fresh_results == train_result["metrics"]

check_rows, check_labels, *_ = rule_data(pd.read_csv(data_file), train_options["seed"], train_options["rows"])

_, held_rows, _, held_labels = train_test_split(
    check_rows, check_labels, test_size=0.25, random_state=train_options["seed"], stratify=check_labels,
)

loaded_accuracy = float(accuracy_score(held_labels, saved_model["model"].predict(held_rows)))

assert abs(loaded_accuracy - fresh_results["accuracy"]) < 1e-12

print({"run": "short training check" if short_run else "full training", "loaded_model_accuracy": loaded_accuracy, "metrics_match": True})

written_files = []

for path in [model_file, metrics_file, check_file]:
    if path.is_relative_to(training_folder):
        shown_name = str(path.relative_to(training_folder))
    else:
        shown_name = path.name

    written_files.append({"file": shown_name, "bytes": path.stat().st_size, "sha256": file_hash(path)})

print(written_files)

{'run': 'short training check', 'loaded_model_accuracy': 0.853, 'metrics_match': True}
[{'file': 'check_runs/package/package_model.joblib', 'bytes': 390648, 'sha256': '170e8b71507d54c61788e5038d2e3258a69d793184e6c6ade8e0cd3242d80be8'}, {'file': 'check_runs/package/package_model_metrics.json', 'bytes': 1339, 'sha256': '1b86dacbbf4823adcb60ab54c593304c550fc725c0348974ff59d20745491abe'}, {'file': 'check_runs/package/package_model_eval.json', 'bytes': 6031, 'sha256': '1bed38ac8fea3a1ca8aaaa635546b04c4f5b2ccce19715b486f865e4075688a3'}]
